# Pipeline Evaluation — chequeo manual post-clasificación

Objetivo: en 5 minutos verificar que las 3 tablas canónicas tienen sentido.  
Corre cada sección de arriba a abajo. Si algo se ve raro, es el lugar para anotarlo.

**Schema:** multi-mention — `reviews_classified.csv` tiene 1 fila por `(review_id, mention_id)`.  
Las métricas de negocio se calculan sobre **menciones** (sentiment, topic, urgency) o sobre **reviews únicas** (total_reviews, avg_rating).

In [1]:
import sys
sys.path.insert(0, '../src')

import pandas as pd

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_rows', 60)

classified = pd.read_csv('../data/processed/reviews_classified.csv')
agg        = pd.read_csv('../data/processed/aggregated.csv')
ins        = pd.read_csv('../data/processed/insights.csv')

# cl = mention rows with successful classification
cl          = classified[classified['sentiment'].notna()].copy()
rating_only = classified[~classified['has_text']].copy()

print(f"reviews_classified : {len(classified):>4} rows  (1 per mention, not per review)")
print(f"  unique reviews   : {classified['review_id'].nunique():>4}")
print(f"  classified rows  : {len(cl)}")
print(f"  rating-only rows : {len(rating_only)}")
print(f"aggregated         : {len(agg):>4} rows")
print(f"insights           : {len(ins):>4} rows")

reviews_classified :  418 rows  (1 per mention, not per review)
  unique reviews   :  250
  classified rows  : 323
  rating-only rows : 94
aggregated         :    4 rows
insights           :   30 rows


---
## 0 · Row vs Review — distinción multi-mention

El CSV tiene más filas que reviews porque cada review con texto genera 1–3 filas (una por topic mencionado).  
Esta sección muestra la expansión y la distribución de menciones.

In [2]:
unique_reviews = classified['review_id'].nunique()
total_rows = len(classified)
expansion = total_rows - unique_reviews

print(f"Unique reviews : {unique_reviews}")
print(f"Total rows     : {total_rows}  (+{expansion} filas extra por expansión multi-mention)")
print()

# Mention distribution — over successfully classified text reviews
mention_dist = (
    cl.groupby('review_id')['mention_id']
    .nunique()
    .value_counts()
    .sort_index()
    .rename_axis('mentions_per_review')
    .rename('review_count')
    .to_frame()
)
mention_dist['pct'] = (mention_dist['review_count'] / mention_dist['review_count'].sum() * 100).round(1)
print("Distribución de menciones (reviews con texto clasificadas exitosamente):")
print(mention_dist.to_string())
print()
total_text = mention_dist['review_count'].sum()
multi_count = mention_dist.loc[mention_dist.index >= 2, 'review_count'].sum()
three_count = mention_dist.loc[mention_dist.index == 3, 'review_count'].sum() if 3 in mention_dist.index else 0
print(f"Reviews con ≥2 menciones : {multi_count} / {total_text}  ({multi_count/total_text*100:.1f}%)  — target: ≥30%")
print(f"Reviews con 3 menciones  : {three_count} / {total_text}  ({three_count/total_text*100:.1f}%)  — target: ≤15%")

Unique reviews : 250
Total rows     : 418  (+168 filas extra por expansión multi-mention)

Distribución de menciones (reviews con texto clasificadas exitosamente):
                     review_count   pct
mentions_per_review                    
1                              44  28.4
2                              54  34.8
3                              57  36.8

Reviews con ≥2 menciones : 111 / 155  (71.6%)  — target: ≥30%
Reviews con 3 menciones  : 57 / 155  (36.8%)  — target: ≤15%


---
## 1 · Sanidad de datos — conteos y nulos

In [3]:
# Vista por fila (menciones) y por review única — dos lentes sobre el mismo dataset
row_view = classified.groupby('business_name').agg(
    total_rows=('review_id', 'count'),
    classified_mentions=('mention_id', lambda s: s.notna().sum()),
).astype(int)

deduped = classified.drop_duplicates(subset='review_id')
review_view = deduped.groupby('business_name').agg(
    unique_reviews=('review_id', 'count'),
    with_text=('has_text', 'sum'),
    pct_classified=('sentiment', lambda s: round(s.notna().mean() * 100, 1)),
)

pd.concat([review_view, row_view], axis=1)

,unique_reviews,with_text,pct_classified,total_rows,classified_mentions
business_name,,,,,
El Mesón de Gauss,50,30,58.0,83,62
IDA Restaurant Bar,100,53,53.0,163,116
Mitica,50,40,80.0,91,81
Vittorio Ristorante,50,33,66.0,81,64


In [4]:
# Nulls en campos de clasificación (solo deberían aparecer en rating-only o clasificaciones fallidas)
cl_fields = ['mention_id', 'sentiment', 'main_topic', 'text_reference', 'urgency', 'is_actionable', 'classification_confidence']
print("Nulls per field (classified rows only):")
print(cl[cl_fields].isna().sum())
print()

# Pares (review_id, mention_id) duplicados?
dupes = classified[['review_id', 'mention_id']].dropna().duplicated().sum()
print(f"Duplicate (review_id, mention_id) pairs: {dupes}")

Nulls per field (classified rows only):
mention_id                   0
sentiment                    0
main_topic                   0
text_reference               0
urgency                      0
is_actionable                0
classification_confidence    0
dtype: int64

Duplicate (review_id, mention_id) pairs: 0


---
## 2 · Distribuciones — sentiment / topic / urgency / confianza

Estas métricas operan sobre **menciones** (cada fila de `cl`), no sobre reviews únicas.

In [5]:
# Sentiment breakdown por negocio — sobre menciones
(
    cl.groupby(['business_name', 'sentiment'])
    .size()
    .unstack(fill_value=0)
    .assign(total=lambda d: d.sum(axis=1))
    .assign(
        pct_pos=lambda d: (d.get('positive', 0) / d['total'] * 100).round(1),
        pct_neg=lambda d: (d.get('negative', 0) / d['total'] * 100).round(1),
    )
)

sentiment,negative,neutral,positive,total,pct_pos,pct_neg
business_name,,,,,,
El Mesón de Gauss,30,1,31,62,50.0,48.4
IDA Restaurant Bar,27,1,88,116,75.9,23.3
Mitica,12,1,68,81,84.0,14.8
Vittorio Ristorante,6,0,58,64,90.6,9.4


In [6]:
# Topic distribution — conteo global de menciones por topic
cl['main_topic'].value_counts()

main_topic
Food Quality              108
Staff Attitude             91
Ambiance                   55
Price / Value              18
Service Speed              18
Overall Experience         16
Menu & Options             14
Booking & Reservations      2
Hygiene & Cleanliness       1
Name: count, dtype: int64

In [7]:
cl['urgency'].value_counts()

urgency
low       270
medium     40
high       13
Name: count, dtype: int64

In [19]:
cl['is_actionable'].value_counts()

is_actionable
True     310
False     13
Name: count, dtype: int64

In [8]:
print(cl['classification_confidence'].describe().round(3))
print(f"\nLow confidence (<0.70): {(cl['classification_confidence'] < 0.70).sum()} mentions")

count    323.000
mean       0.932
std        0.036
min        0.750
25%        0.920
50%        0.950
75%        0.950
max        0.980
Name: classification_confidence, dtype: float64

Low confidence (<0.70): 0 mentions


---
## 3 · Spot-check — muestras aleatorias para revisión manual

Leé cada texto y verificá que el label tenga sentido. Si algo está mal anotalo más abajo.

In [9]:
SAMPLE_N    = 20
RANDOM_SEED = 42

sample = cl.sample(SAMPLE_N, random_state=RANDOM_SEED)[
    ['business_name', 'clean_text', 'mention_id', 'sentiment', 'main_topic',
     'text_reference', 'urgency', 'is_actionable', 'classification_confidence']
].reset_index(drop=True)

sample

,business_name,clean_text,mention_id,sentiment,main_topic,text_reference,urgency,is_actionable,classification_confidence
0,El Mesón de Gauss,"El lugar muy lindo, la comida buena, la atención hace que no tengas ganas de volver....",3.0,negative,Staff Attitude,la atención hace que no tengas ganas de volver,high,True,0.92
1,El Mesón de Gauss,"Lindo ambiente, buena comida, buena atención",2.0,positive,Food Quality,buena comida,low,True,0.95
2,Mitica,"Fuimos 2 personas comimos el menú que parecía estar bien,pero la entrada ,un tenedor sobraba La carne de cerdo sin g...",1.0,negative,Food Quality,"La carne de cerdo sin gusto con una salsa incomible,( con las salsas agridulces baratas para hacer ,gran decepcion",medium,True,0.95
3,IDA Restaurant Bar,"Hermoso lugar, muy rica la comida y un ambiente con buena música 🎶",1.0,positive,Ambiance,"Hermoso lugar, muy rica la comida y un ambiente con buena música 🎶",low,True,0.95
4,IDA Restaurant Bar,Hermoso lugar. Muy rica la comida. Poco variedad de menu,2.0,positive,Food Quality,Muy rica la comida.,low,True,0.95
5,El Mesón de Gauss,"Una pena que un lugar tan lindo, tenga mala atención. La moza muy amable pero se olvidaron de comandar el pedido, po...",1.0,negative,Service Speed,"se olvidaron de comandar el pedido, por lo cual estuvimos 1h esperando el almuerzo",medium,True,0.95
6,Mitica,"Todo es excelente. La atención, la carta y la comida.",1.0,positive,Food Quality,la comida,low,True,0.95
7,El Mesón de Gauss,"Fuimos con los peques a almorzar, tuvimos una experiencia hermosa. La moza muy atenta, rica la comida y el precio ac...",1.0,positive,Staff Attitude,La moza muy atenta,low,True,0.95
8,Mitica,Excelente comida Muy buena atencion de Paula Recomiendo!,2.0,positive,Staff Attitude,Muy buena atencion de Paula,low,True,0.92
9,Vittorio Ristorante,Fui desde el primer día....y lo sigo eligiendo,1.0,positive,Overall Experience,Fui desde el primer día....y lo sigo eligiendo,low,True,0.95


In [24]:
# text_reference quality — ¿las citas son literales?
# Mismo criterio que _check_text_reference en classification/__init__.py: word-level overlap ≥ 0.8
import re

def clean_words(s):
    return re.findall(r'\b\w+\b', str(s).lower())

def word_overlap(ref, text):
    words = clean_words(ref)
    if not words:
        return 0.0
    text_words = set(clean_words(text))
    return round(sum(1 for w in words if w in text_words) / len(words), 2)

sample_refs = cl[cl['text_reference'].notna()].sample(10, random_state=42)[
    ['review_id', 'clean_text', 'main_topic', 'text_reference']
].reset_index(drop=True)

sample_refs['overlap'] = sample_refs.apply(
    lambda r: word_overlap(r['text_reference'], r['clean_text']), axis=1
)

for _, r in sample_refs.iterrows():
    flag = "✓" if r['overlap'] >= 0.8 else "⚠"
    print(f"{flag} overlap={r['overlap']}  [{r['main_topic']}]")
    print(f"   ref : {r['text_reference']!r}")
    print(f"   text: {r['clean_text'][:120]!r}")
    print()

✓ overlap=1.0  [Staff Attitude]
   ref : 'la atención hace que no tengas ganas de volver'
   text: 'El lugar muy lindo, la comida buena, la atención hace que no tengas ganas de volver....'

✓ overlap=1.0  [Food Quality]
   ref : 'buena comida'
   text: 'Lindo ambiente, buena comida, buena atención'

✓ overlap=1.0  [Food Quality]
   ref : 'La carne de cerdo sin gusto con una salsa incomible,( con las salsas agridulces baratas para hacer ,gran decepcion'
   text: 'Fuimos 2 personas comimos el menú que parecía estar bien,pero la entrada ,un tenedor sobraba La carne de cerdo sin gusto'

✓ overlap=1.0  [Ambiance]
   ref : 'Hermoso lugar, muy rica la comida y un ambiente con buena música 🎶'
   text: 'Hermoso lugar, muy rica la comida y un ambiente con buena música 🎶'

✓ overlap=1.0  [Food Quality]
   ref : 'Muy rica la comida.'
   text: 'Hermoso lugar. Muy rica la comida. Poco variedad de menu'

✓ overlap=1.0  [Service Speed]
   ref : 'se olvidaron de comandar el pedido, por lo cual estuvimo

In [25]:
# LOW CONFIDENCE — revisar estos primero (más propensos a error)
cl[cl['classification_confidence'] < 0.75][
    ['business_name', 'clean_text', 'sentiment', 'main_topic', 'urgency', 'classification_confidence']
].sort_values('classification_confidence').head(15)

,business_name,clean_text,sentiment,main_topic,urgency,classification_confidence


In [12]:
# HIGH URGENCY — verificar que realmente sean urgentes
cl[cl['urgency'] == 'high'][
    ['business_name', 'clean_text', 'sentiment', 'main_topic', 'text_reference', 'urgency', 'is_actionable']
]

,business_name,clean_text,sentiment,main_topic,text_reference,urgency,is_actionable
168,IDA Restaurant Bar,La comida más o menos y lo peor el servicio dejó muchísimo que desear. Tuvieron un accidente con un plato de bondiol...,negative,Staff Attitude,nos empaparon con el jugo de la cocción. Nadie se acercó a pedir disculpas ni a ofrecer una solución.,high,True
170,IDA Restaurant Bar,"Fuimos un viernes por la noche con mi mamá, planeando una linda ""salida de chicas"", pero lamentablemente terminamos ...",negative,Food Quality,"me sirvieron su guarnición de boniato literalmente podrido, con gusto a tierra que provocaba náuseas al masticar",high,True
173,IDA Restaurant Bar,"Muy mala experiencia, gastarse 100.000 en una salida para no ser bien atendido es un desprecio a tu plata",negative,Staff Attitude,no ser bien atendido es un desprecio a tu plata,high,True
179,IDA Restaurant Bar,"La verdad es que vivo fuera del país, y cada año que vengo, me encanta volver a los lugares donde he tendido muy lin...",negative,Staff Attitude,"el servício ha sido malísimo, muy mala actitud, sin ganas de atenderte",high,True
190,IDA Restaurant Bar,Mala atención,negative,Staff Attitude,Mala atención,high,True
202,IDA Restaurant Bar,"Acotaron mucho la carta, y encima lo que pedíamos no tenían. Mucha demora en traernos las cosas. Casi nada de varied...",negative,Staff Attitude,"Cuando se la pedí al mozo me dijo: agua no regalamos, queres agua de la canilla?",high,True
216,El Mesón de Gauss,"Un horror,fui con mi hija y mi nieta a merendar,dos cafe con leche,tostado tostadas fit,y criollitos,en la mitad de ...",negative,Food Quality,mi hija toma un sorbo del cafe y se saca de la boca una mosca muerta!!!!! Q asco,high,True
218,El Mesón de Gauss,"Un horror,fui con mi hija y mi nieta a merendar,dos cafe con leche,tostado tostadas fit,y criollitos,en la mitad de ...",negative,Hygiene & Cleanliness,se ve q ni lavan ni miran las tazas!! Vomito del asco,high,True
245,El Mesón de Gauss,"Anoche después de ir al teatro fuimos a comer al Mesón, pedimos lomito se demoraron una hora en traerlo, no nos ofre...",negative,Staff Attitude,"La moza nos dijo que ya lo arreglaba y nunca más apareció, no nos trajeron más pan ni nada",high,True
268,El Mesón de Gauss,"El lugar muy lindo, la comida buena, la atención hace que no tengas ganas de volver....",negative,Staff Attitude,la atención hace que no tengas ganas de volver,high,True


In [13]:
# NEGATIVE + ACTIONABLE — los más importantes para el reporte
cl[(cl['sentiment'] == 'negative') & (cl['is_actionable'] == True)][
    ['business_name', 'clean_text', 'main_topic', 'text_reference', 'urgency', 'classification_confidence']
].sort_values('urgency', ascending=False)

,business_name,clean_text,main_topic,text_reference,urgency,classification_confidence
139,IDA Restaurant Bar,"No fuimos bien atendidos, nos tomaron la mitad del pedido, al no haber opciones de platos principales vegetarianos, ...",Staff Attitude,"No fuimos bien atendidos, nos tomaron la mitad del pedido",medium,0.85
209,IDA Restaurant Bar,"Creo que fui un mal día porque me habían hecho buenos comentarios. Estuvo todo bastante lerdo, una burrata congelada...",Price / Value,pedimos uno más nos dijeron que lo cobraban extra,medium,0.85
213,El Mesón de Gauss,"Una pena que un lugar tan lindo, tenga mala atención. La moza muy amable pero se olvidaron de comandar el pedido, po...",Service Speed,"se olvidaron de comandar el pedido, por lo cual estuvimos 1h esperando el almuerzo",medium,0.95
221,El Mesón de Gauss,"Buena comida, lindo ambiente y lugar fuimos un día que justo no tenían su famoso lomito con pan de pizza y el otro e...",Price / Value,el otro era medio común para cobrarlo tan bien,medium,0.85
229,El Mesón de Gauss,"Vinimos con unos amigos, y NO HAY PIZZAS! lo peor es que no te advierte cuando te indican que abras la carta. Y tamp...",Service Speed,Sinceramente el servicio ha decaído considerablemente.,medium,0.88
...,...,...,...,...,...,...
245,El Mesón de Gauss,"Anoche después de ir al teatro fuimos a comer al Mesón, pedimos lomito se demoraron una hora en traerlo, no nos ofre...",Staff Attitude,"La moza nos dijo que ya lo arreglaba y nunca más apareció, no nos trajeron más pan ni nada",high,0.92
170,IDA Restaurant Bar,"Fuimos un viernes por la noche con mi mamá, planeando una linda ""salida de chicas"", pero lamentablemente terminamos ...",Food Quality,"me sirvieron su guarnición de boniato literalmente podrido, con gusto a tierra que provocaba náuseas al masticar",high,0.95
202,IDA Restaurant Bar,"Acotaron mucho la carta, y encima lo que pedíamos no tenían. Mucha demora en traernos las cosas. Casi nada de varied...",Staff Attitude,"Cuando se la pedí al mozo me dijo: agua no regalamos, queres agua de la canilla?",high,0.98
268,El Mesón de Gauss,"El lugar muy lindo, la comida buena, la atención hace que no tengas ganas de volver....",Staff Attitude,la atención hace que no tengas ganas de volver,high,0.92


---
## 4 · Aggregated — chequeo de métricas de negocio

In [14]:
agg

,business_name,total_reviews,reviews_with_text,avg_rating,pct_positive,pct_neutral,pct_negative,top_topic_1,top_topic_2,top_topic_3,high_urgency_count,actionable_count
0,El Mesón de Gauss,50,30,3.6,50.0,1.6,48.4,Food Quality,Staff Attitude,Ambiance,5,60
1,IDA Restaurant Bar,100,53,4.4,75.9,0.9,23.3,Food Quality,Staff Attitude,Ambiance,6,111
2,Mitica,50,40,4.6,84.0,1.2,14.8,Staff Attitude,Food Quality,Ambiance,0,78
3,Vittorio Ristorante,50,33,4.6,90.6,0.0,9.4,Food Quality,Staff Attitude,Ambiance,2,61


In [15]:
# Verificar que pct_positive + pct_neutral + pct_negative ≈ 100 en cada fila
check = agg.copy()
check['pct_sum'] = check['pct_positive'] + check['pct_neutral'] + check['pct_negative']
print("pct sum per business (expected ~100):")
print(check[['business_name', 'pct_sum']].to_string(index=False))

pct sum per business (expected ~100):
      business_name  pct_sum
  El Mesón de Gauss    100.0
 IDA Restaurant Bar    100.1
             Mitica    100.0
Vittorio Ristorante    100.0


---
## 5 · Insights — ranking de problemas

In [16]:
# Top issues por negocio — estas son las recomendaciones del reporte
ins.sort_values('priority_score', ascending=False)

,business_name,main_topic,mention_count,pct_negative,avg_urgency_score,actionable_count,priority_score
0,El Mesón de Gauss,Food Quality,22,45.5,1.27,22,12.71
1,El Mesón de Gauss,Service Speed,7,85.7,1.86,7,11.16
2,IDA Restaurant Bar,Staff Attitude,32,25.0,1.38,32,11.04
3,IDA Restaurant Bar,Service Speed,6,83.3,1.83,6,9.15
4,El Mesón de Gauss,Staff Attitude,12,41.7,1.67,12,8.36
5,IDA Restaurant Bar,Price / Value,5,80.0,1.80,5,7.20
6,IDA Restaurant Bar,Menu & Options,5,100.0,1.20,5,6.00
7,Vittorio Ristorante,Service Speed,4,75.0,2.00,4,6.00
8,El Mesón de Gauss,Price / Value,8,50.0,1.25,7,5.00
9,IDA Restaurant Bar,Food Quality,37,10.8,1.14,37,4.56


In [17]:
# Por negocio — qué problema lidera en cada uno
ins.groupby('business_name').first()[['main_topic', 'priority_score', 'pct_negative', 'avg_urgency_score']]

,main_topic,priority_score,pct_negative,avg_urgency_score
business_name,,,,
El Mesón de Gauss,Food Quality,12.71,45.5,1.27
IDA Restaurant Bar,Staff Attitude,11.04,25.0,1.38
Mitica,Food Quality,3.21,10.7,1.07
Vittorio Ristorante,Service Speed,6.00,75.0,2.00


---
## 6 · Hallazgos manuales

Usá esta celda para anotar lo que encontraste en el spot-check:

**Clasificaciones correctas:** 

**Clasificaciones dudosas o incorrectas:** 

**Patrones que el modelo no captó bien:** 

**Acciones a tomar (prompt, taxonomía, etc.):** 